# ESRGAN Demo: Image Upscaling

This notebook demonstrates how to use the trained ESRGAN model to upscale images 4×.

**Requirements:** Run from the repository root after `pip install -r requirements.txt`.

In [ ]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Ensure src is importable when running from notebooks/
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.rrdb_net import RRDBNet
from src.inference import run_inference, tiled_upscale
from src.utils import load_image, save_image, compute_psnr, compute_ssim, tensor_to_numpy

print('Imports OK')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 1. Configuration

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────
CHECKPOINT_PATH = '../checkpoints/final.pth'  # Set to your checkpoint
INPUT_IMAGE     = '../data/sample/example.png' # Set to your LR image
OUTPUT_IMAGE    = '/tmp/esrgan_sr_output.png'

SCALE       = 4
TILE_SIZE   = 128   # LR tile size
TILE_OVERLAP = 16   # Overlap in pixels

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Load Model

In [ ]:
# Build generator
model = RRDBNet(in_nc=3, out_nc=3, nf=64, nb=23, gc=32, scale=SCALE).to(device)
model.eval()

# Load weights if checkpoint exists
if os.path.isfile(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    state_dict = ckpt.get('generator', ckpt)
    model.load_state_dict(state_dict)
    print(f'Loaded checkpoint: {CHECKPOINT_PATH}')
else:
    print(f'Checkpoint not found at {CHECKPOINT_PATH} — using random weights for demo')

total_params = sum(p.numel() for p in model.parameters())
print(f'Generator parameters: {total_params:,}')

## 3. Run Inference

In [ ]:
# Create a synthetic example if no image is available
if not os.path.isfile(INPUT_IMAGE):
    print(f'Input not found; generating synthetic LR image for demo')
    synthetic = np.random.randint(64, 200, (32, 32, 3), dtype=np.uint8)
    # Add some structure
    for i in range(0, 32, 8):
        synthetic[i:i+4, :] = [200, 100, 50]
    lr_pil = Image.fromarray(synthetic)
    os.makedirs(os.path.dirname(INPUT_IMAGE) if os.path.dirname(INPUT_IMAGE) else '.', exist_ok=True)
    lr_pil.save(INPUT_IMAGE)

# Load input image
lr_img = load_image(INPUT_IMAGE, device=device)
print(f'Input shape: {lr_img.shape}  ({lr_img.shape[2]}x{lr_img.shape[1]} px)')

# Run tiled inference
cfg = {
    'model': {'in_nc': 3, 'out_nc': 3, 'nf': 64, 'nb': 23, 'gc': 32, 'scale': SCALE, 'checkpoint': CHECKPOINT_PATH},
    'io': {'input': INPUT_IMAGE, 'output': '/tmp'},
    'tiling': {'enabled': True, 'tile_size': TILE_SIZE, 'tile_overlap': TILE_OVERLAP},
}
sr_img = run_inference(model, lr_img.cpu(), cfg, device)
print(f'Output shape: {sr_img.shape}  ({sr_img.shape[2]}x{sr_img.shape[1]} px)')

# Save output
save_image(sr_img, OUTPUT_IMAGE)
print(f'Saved SR image to: {OUTPUT_IMAGE}')

## 4. Visualize Results

In [ ]:
lr_np  = tensor_to_numpy(lr_img.cpu())
sr_np  = tensor_to_numpy(sr_img)

# Bicubic upscale for comparison
lr_pil = Image.fromarray(lr_np)
bicubic = lr_pil.resize((lr_pil.width * SCALE, lr_pil.height * SCALE), Image.BICUBIC)
bicubic_np = np.array(bicubic)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(lr_np)
axes[0].set_title(f'LR Input ({lr_pil.width}×{lr_pil.height})', fontsize=13)
axes[0].axis('off')

axes[1].imshow(bicubic_np)
axes[1].set_title(f'Bicubic ×{SCALE}', fontsize=13)
axes[1].axis('off')

axes[2].imshow(sr_np)
axes[2].set_title(f'ESRGAN ×{SCALE}', fontsize=13)
axes[2].axis('off')

plt.suptitle('Super-Resolution Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Comparison saved to /tmp/comparison.png')

## 5. Quality Metrics (if HR reference available)

In [ ]:
HR_REFERENCE = None  # Set to path of HR reference image if available

if HR_REFERENCE and os.path.isfile(HR_REFERENCE):
    hr_tensor = load_image(HR_REFERENCE)
    # Resize SR to match HR if needed
    import torch.nn.functional as F
    if hr_tensor.shape != sr_img.shape:
        sr_resized = F.interpolate(sr_img.unsqueeze(0), size=hr_tensor.shape[-2:], mode='bicubic').squeeze(0)
    else:
        sr_resized = sr_img

    psnr = compute_psnr(sr_resized, hr_tensor)
    ssim = compute_ssim(sr_resized, hr_tensor)
    print(f'PSNR: {psnr:.2f} dB')
    print(f'SSIM: {ssim:.4f}')
else:
    print('No HR reference available. Set HR_REFERENCE to compute PSNR/SSIM.')